In [1]:
import pandas as pd
from pathlib import Path

In [2]:
load_file = "nl20_load_timeseries_per_bus.csv"
cf_file = "nl20_renewable_cf_per_generator.csv"

In [3]:
rep_day_versions = [2, 4, 8, 16, 30]

# weights
load_weight = 1.0
cf_weight = 1.0

# timestamp column in files
timestamp_col = "snapshot"


In [4]:
load_cols = pd.read_csv(load_file, nrows=0).columns.tolist()
cf_cols = pd.read_csv(cf_file, nrows=0).columns.tolist()

# remove timestamp column
load_series_cols = [c for c in load_cols if c != timestamp_col]
cf_series_cols = [c for c in cf_cols if c != timestamp_col]


In [5]:
# make YAML-safe series names
def clean_name(prefix: str, col: str) -> str:
    name = col.replace("-", "_").replace(" ", "_").replace("/", "_")
    return f"{prefix}_{name}"


In [6]:
def build_time_series_block() -> str:
    lines = []
    lines.append("time_series:")
    lines.append("  default:")
    lines.append('    csv_options:')
    lines.append('      delim: ","')
    lines.append(f'    timestamp: "{timestamp_col}"')
    lines.append('    sampling_time: "Hour(1)"')
    lines.append("")
    
    # load series
    for col in load_series_cols:
        series_name = clean_name("Load", col)
        lines.append(f"  {series_name}:")
        lines.append(f'    source: "{load_file}"')
        lines.append(f'    value_column: "{col}"')
        lines.append(f"    weight: {load_weight}")
        lines.append("")
    
    # cf series
    for col in cf_series_cols:
        series_name = clean_name("CF", col)
        lines.append(f"  {series_name}:")
        lines.append(f'    source: "{cf_file}"')
        lines.append(f'    value_column: "{col}"')
        lines.append(f"    weight: {cf_weight}")
        lines.append("")
    
    return "\n".join(lines)

time_series_block = build_time_series_block()

In [9]:
for n_rep in rep_day_versions:
    yaml_text = f"""method:
  options:
    total_periods: 365
    representative_periods: {n_rep}
    time_steps_per_period: 24
    sampling_time: "Hour(1)"
    mandatory_periods: []

  optimization:
    integral_weights: false
    binary_ordering: true
    equal_weights: true

    duration_curve_error:
      weight: 1.0
      number_bins: 10
      type: "absolute"

results:
  save_results: true
  result_dir: "repdays_{n_rep}"
  create_plots: false

{time_series_block}
"""

    out_file = Path(f"rep_days_{n_rep}.yaml")
    out_file.write_text(yaml_text, encoding="utf-8")

print("Created YAML files:")
for n_rep in rep_day_versions:
    print(f"  rep_days_{n_rep}.yaml")

Created YAML files:
  rep_days_2.yaml
  rep_days_4.yaml
  rep_days_8.yaml
  rep_days_16.yaml
  rep_days_30.yaml
